# Imports

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# Current Registry

In [3]:
registry_df = pd.read_csv("current_rental_registrations_251001.csv")

In [4]:
registry_df["formatted_address"] = registry_df["RegisteredAddress"].str.replace(r"\s+,", ",", regex=True)

In [5]:
registry_df["formatted_address"] = registry_df["formatted_address"].astype(str).str.upper().str.strip()

In [6]:
# Remove unit numbers before the first comma (e.g., " AVE 5," → " AVE,")
registry_df["formatted_address"] = registry_df["formatted_address"].str.replace(
    r"\s+\d+(?=,)", "", regex=True
).str.strip()


In [7]:
import re

def clean_units(address):
    # Match pattern: [street] [unit], [city], MA [ZIP]
    match = re.match(r"^(.*\b(?:ST|AV|AVE|RD|BLVD|PL|CT|DR|TER|WAY|LN|SQ|TE|CIR|PKWY|PLZ|HWY))\s+[A-Z0-9\-]+, (.+?, MA \d{5})$", address)
    if match:
        return f"{match.group(1)}, {match.group(2)}"
    return address

registry_df["formatted_address"] = registry_df["formatted_address"].apply(clean_units)


In [8]:
registry_df['formatted_address'].sample(30)

32208           233 HANCOCK ST, DORCHESTER, MA 02125
23733           17 ROCKINGHAM RD, MATTAPAN, MA 02126
37238          29 LYNDHURST ST, DORCHESTER, MA 02124
26959           33 DELLE AVE, MISSION HILL, MA 02120
30723               30 GARRISON ST, BOSTON, MA 02116
19094                42 CHAUNCY ST, BOSTON, MA 02111
15971      14-16 ALHAMBRA RD, WEST ROXBURY, MA 02132
27419      371 DORCHESTER ST, SOUTH BOSTON, MA 02127
35803              90 KILSYTH RD, BRIGHTON, MA 02135
18381            131 CENTRE ST, DORCHESTER, MA 02124
39430             102 MORELAND ST, ROXBURY, MA 02119
4965                 520 BEACON ST, BOSTON, MA 02215
19699       185 CHESTNUT HILL AV, BRIGHTON, MA 02135
23253       1625 COMMONWEALTH AV, BRIGHTON, MA 02135
25550       10-8 COPENGER ST, MISSION HILL, MA 02120
22257            343 COMMERCIAL ST, BOSTON, MA 02109
29953    94 FOREST HILLS ST, JAMAICA PLAIN, MA 02130
25286                1 FRANKLIN ST, BOSTON, MA 02110
28857              275 EVERETT ST, ALLSTON, MA

# Assessor Dataset

In [9]:
assessment_df = pd.read_csv("fy2025-property-assessment-data_12_30_2024.csv", dtype={21: str}, low_memory=False)

In [10]:
def combine_street_numbers(row):
    try:
        st_num = str(int(float(row['ST_NUM']))) if pd.notnull(row['ST_NUM']) else ""
        st_num2 = str(int(float(row['ST_NUM2']))) if pd.notnull(row['ST_NUM2']) else ""
        return f"{st_num}-{st_num2}" if st_num and st_num2 else st_num
    except:
        return ""

assessment_df["ST_NUM_COMBINED"] = assessment_df.apply(combine_street_numbers, axis=1)


In [11]:
assessment_df["ZIP_CODE_CLEAN"] = assessment_df["ZIP_CODE"].astype(str).str.extract(r'(\d+)')[0].str.zfill(5)

assessment_df["ST_NAME_CLEAN"] = assessment_df["ST_NAME"].astype(str).str.replace(r'\bAVE\.', 'AV', regex=True)

assessment_df["formatted_address"] = (
    assessment_df["ST_NUM_COMBINED"].str.strip() + " " +
    assessment_df["ST_NAME_CLEAN"].astype(str).str.strip() + ", " +
    assessment_df["CITY"].astype(str).str.strip() + ", MA " +
    assessment_df["ZIP_CODE_CLEAN"]
).str.upper()


In [12]:
assessment_df["formatted_address"].dropna().unique()[:10]

array(['104 PUTNAM ST, EAST BOSTON, MA 02128',
       '197 LEXINGTON ST, EAST BOSTON, MA 02128',
       '199 LEXINGTON ST, EAST BOSTON, MA 02128',
       '201 LEXINGTON ST, EAST BOSTON, MA 02128',
       '203 LEXINGTON ST, EAST BOSTON, MA 02128',
       '205-207 LEXINGTON ST, EAST BOSTON, MA 02128',
       '209-211 LEXINGTON ST, EAST BOSTON, MA 02128',
       '213 LEXINGTON ST, EAST BOSTON, MA 02128',
       '215 LEXINGTON ST, EAST BOSTON, MA 02128',
       '217 LEXINGTON ST, EAST BOSTON, MA 02128'], dtype=object)

In [13]:
# Show side-by-side original and formatted addresses from a sample
assessment_df[["ST_NUM", "ST_NUM2", "ST_NAME", "CITY", "ZIP_CODE", "formatted_address"]].sample(50)


,ST_NUM,ST_NUM2,ST_NAME,CITY,ZIP_CODE,formatted_address
115971,5.0,7.0,ADANAC TE,DORCHESTER,2124.0,"5-7 ADANAC TE, DORCHESTER, MA 02124"
98427,5.0,NaN,EMROSE TE,DORCHESTER,2125.0,"5 EMROSE TE, DORCHESTER, MA 02125"
81020,656.0,NaN,Tremont ST,BOSTON,2118.0,"656 TREMONT ST, BOSTON, MA 02118"
161272,68.0,NaN,CHESBROUGH RD,WEST ROXBURY,2132.0,"68 CHESBROUGH RD, WEST ROXBURY, MA 02132"
98704,64.0,66.0,MONADNOCK ST,DORCHESTER,2125.0,"64-66 MONADNOCK ST, DORCHESTER, MA 02125"
176264,14.0,NaN,WESTFORD ST,ALLSTON,2134.0,"14 WESTFORD ST, ALLSTON, MA 02134"
93454,370.0,NaN,ARBORWAY ST,JAMAICA PLAIN,2130.0,"370 ARBORWAY ST, JAMAICA PLAIN, MA 02130"
9279,220.0,NaN,Paris ST,EAST BOSTON,2128.0,"220 PARIS ST, EAST BOSTON, MA 02128"
6433,11.0,NaN,LAMSON ST,EAST BOSTON,2128.0,"11 LAMSON ST, EAST BOSTON, MA 02128"
129787,316.0,NaN,WOOD AV,HYDE PARK,2136.0,"316 WOOD AV, HYDE PARK, MA 02136"


# Current vs Assessor

In [14]:
# Compare formatted FY2025 addresses to registry addresses
unregistered_props = assessment_df[~assessment_df["formatted_address"].isin(registry_df["formatted_address"])]

# Count how many are unregistered
unregistered_count = unregistered_props.shape[0]

# Total properties in FY2025
total_properties = assessment_df.shape[0]

# Calculate percentage unregistered
unregistered_pct = (unregistered_count / total_properties) * 100

unregistered_count, total_properties, round(unregistered_pct, 2)

(134491, 183445, 73.31)

In [15]:
registered_properties_from_assessment = assessment_df[
    assessment_df["formatted_address"].isin(registry_df["formatted_address"])
]

# Show first 10 matching addresses
registered_properties_from_assessment["formatted_address"].sample(30)

73529          549 E FOURTH ST, SOUTH BOSTON, MA 02127
59737           245 W FIFTH ST, SOUTH BOSTON, MA 02127
55741            31 MASSACHUSETTS AV, BOSTON, MA 02115
70943           182 W NINTH ST, SOUTH BOSTON, MA 02127
97320            135 TOWNSEND ST, DORCHESTER, MA 02121
122905    34-36 PLEASANT HILL AV, DORCHESTER, MA 02124
182871              144 KENRICK ST, BRIGHTON, MA 02135
19622                 9 HAWTHORNE PL, BOSTON, MA 02114
50294             135 MARLBOROUGH ST, BOSTON, MA 02116
37221                  23 HOLYOKE ST, BOSTON, MA 02116
98170              38 FAYSTON ST, DORCHESTER, MA 02121
172753           146 SUTHERLAND RD, BRIGHTON, MA 02135
22818                   21 BEACON ST, BOSTON, MA 02108
88291                  67 CENTRE ST, ROXBURY, MA 02119
181981                 19 SOUTH ST, BRIGHTON, MA 02135
107404                93 ORMOND ST, MATTAPAN, MA 02126
91191              28 GLEN RD, JAMAICA PLAIN, MA 02130
105781            35 THEODORE ST, DORCHESTER, MA 02124
88408     

# 311 Service Request

In [16]:
service_df = pd.read_csv("dff4d804-5031-443a-8409-8344efd0e5c8.csv", low_memory=False)

In [17]:
# Known city/neighborhood names to catch multi-word places like "SOUTH BOSTON", "JAMAICA PLAIN"
boston_neighborhoods = [
    "SOUTH BOSTON", "EAST BOSTON", "JAMAICA PLAIN", "MATTAPAN", "ROXBURY", 
    "BRIGHTON", "CHARLESTOWN", "HYDE PARK", "DORCHESTER", "WEST ROXBURY", 
    "ALLSTON", "ROSLINDALE", "BACK BAY", "FENWAY", "MISSION HILL", "NORTH END",
    "SOUTH END", "CHINATOWN"
]

def format_location(location):
    try:
        parts = location.strip().split()
        if len(parts) < 4:
            return location.upper()

        zip_code = parts[-1]
        state = parts[-2]

        # Try 2-word city names first
        possible_city = " ".join(parts[-4:-2]).upper()
        if possible_city in boston_neighborhoods:
            city = possible_city
            street = " ".join(parts[:-4])
        else:
            # Fall back to 1-word city names
            city = parts[-3].upper()
            street = " ".join(parts[:-3])

        return f"{street}, {city}, {state} {zip_code}".upper()
    except:
        return ""
service_df["formatted_address"] = service_df["location"].astype(str).apply(format_location)

# Replace AVE (or AVE.) with AV in 311 formatted addresses
service_df["formatted_address"] = service_df["formatted_address"].str.replace(
    r"\bAVE\.?\b", "AV", regex=True
)


service_df["formatted_address"].sample(30)

72986     INTERSECTION OF CONSTITUTION PLZ & CONSTITUTIO...
109838              26 SUNNYBANK RD, WEST ROXBURY, MA 02132
279396                   18 BAILEY ST, DORCHESTER, MA 02124
105856                  66 WILLIAMS AV, HYDE PARK, MA 02136
45724              5 SAINT MARK ST, JAMAICA PLAIN, MA 02130
65814                    126 THORNTON ST, ROXBURY, MA 02119
211762     3686-3688 WASHINGTON ST, JAMAICA PLAIN, MA 02130
274291                     79 N MARGIN ST, BOSTON, MA 02113
255960    INTERSECTION OF STRATHMORE RD & SUTHERLAND, RD...
115481                      44 BURBANK ST, BOSTON, MA 02115
48146     INTERSECTION OF PROVINCE ST & SCHOOL, ST, BOST...
183216                    34 KINROSS RD, BRIGHTON, MA 02135
202641                       17 IRVING ST, BOSTON, MA 02114
217415                 49 STELLMAN RD, ROSLINDALE, MA 02131
50977                 7A HALF MOON ST, DORCHESTER, MA 02125
273517    INTERSECTION OF VICKSBURG ST & E FIRST ST, SOU...
92433                         75 CANAL S

# Compare Current vs 311 

In [18]:
unregistered_service_requests = service_df[~service_df["formatted_address"].isin(registry_df["formatted_address"])]

unregistered_service_count = unregistered_service_requests.shape[0]
total_service_requests = service_df.shape[0]
unregistered_service_pct = (unregistered_service_count / total_service_requests) * 100

unregistered_service_count, total_service_requests, round(unregistered_service_pct, 2)


(229914, 282836, 81.29)

In [19]:
registered_service_requests = service_df[service_df["formatted_address"].isin(registry_df["formatted_address"])]

registered_service_requests["formatted_address"].sample(30)

230610          39 FAWNDALE RD, ROSLINDALE, MA 02131
59416           284 SUMNER ST, EAST BOSTON, MA 02128
229218              312 WARREN ST, ROXBURY, MA 02119
124330              11 HOBSON ST, BRIGHTON, MA 02135
174362             15 N BEACON ST, ALLSTON, MA 02134
187409               84 GORDON ST, ALLSTON, MA 02135
179179            18 WINCHESTER ST, BOSTON, MA 02116
137346              32 GERALD RD, BRIGHTON, MA 02135
101230                  53 HULL ST, BOSTON, MA 02113
254554               5 APPLETON ST, BOSTON, MA 02116
38011     170-172 MAVERICK ST, EAST BOSTON, MA 02128
156314               93 CHARTER ST, BOSTON, MA 02113
59402       7-9 SPALDING ST, JAMAICA PLAIN, MA 02130
186483            115 DEERING RD, MATTAPAN, MA 02126
238415                 10 UNITY ST, BOSTON, MA 02113
78196         88 W SPRINGFIELD ST, ROXBURY, MA 02118
34629                 7 DUVAL ST, BRIGHTON, MA 02135
332              127 SELWYN ST, ROSLINDALE, MA 02131
198319         21 WENHAM ST, JAMAICA PLAIN, MA

# Unregistered list

In [24]:
unregistered_service_addrs = service_df[~service_df["formatted_address"].isin(registry_df["formatted_address"])]["formatted_address"].dropna().unique()

unregistered_assessor_addrs = assessment_df[~assessment_df["formatted_address"].isin(registry_df["formatted_address"])]["formatted_address"].dropna().unique()

# Combine and sort
all_unregistered_addresses = sorted(set(unregistered_service_addrs).union(set(unregistered_assessor_addrs)))


In [25]:
for addr in all_unregistered_addresses[:10]:
    print(addr)

 
 A ST, BOSTON, MA 02210
 A ST, SOUTH BOSTON, MA 02127
 ABBY RD, BRIGHTON, MA 02135
 ACADEMY HILL RD, BRIGHTON, MA 02135
 ACADIA ST, SOUTH BOSTON, MA 02127
 ACORN ST, BOSTON, MA 02108
 ACTON ST, HYDE PARK, MA 02136
 ADA ST, ROSLINDALE, MA 02131
 ADAMS PL, BOSTON, MA 02114
